In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib

print("🚨 Starting Model 3 Training: Unsupervised Anomaly Detector (Isolation Forest)")

# ==========================================
# 1. GENERATE MOCK DATA (Normal + Anomalies)
# ==========================================
np.random.seed(42)
num_normal = 4900
num_anomalies = 100 # 2% of transactions are weird/fraudulent

# Generate NORMAL transactions
normal_incomes = np.random.choice([2500, 3500, 5000, 8000], num_normal)
normal_avg_spends = normal_incomes * np.random.uniform(0.01, 0.05, num_normal) # Normal meal/shopping
normal_amounts = normal_avg_spends * np.random.uniform(0.8, 1.2, num_normal) # Slight variance

# Generate ANOMALY transactions (e.g., hacked account, sudden impulse buy)
anomaly_incomes = np.random.choice([2500, 3500, 5000, 8000], num_anomalies)
anomaly_avg_spends = anomaly_incomes * np.random.uniform(0.01, 0.05, num_anomalies)
# The anomaly! Spending 10x to 50x more than their usual average
anomaly_amounts = anomaly_avg_spends * np.random.uniform(10, 50, num_anomalies) 

# Combine them into a DataFrame
df = pd.DataFrame({
    'monthly_income': np.concatenate([normal_incomes, anomaly_incomes]),
    'user_avg_category_spend': np.concatenate([normal_avg_spends, anomaly_avg_spends]),
    'transaction_amount': np.concatenate([normal_amounts, anomaly_amounts])
})

# ==========================================
# 2. FEATURE ENGINEERING (The Secret Sauce)
# ==========================================
# The model doesn't care about the raw RM amount. It cares about the RATIOS.
df['amount_to_income_ratio'] = df['transaction_amount'] / df['monthly_income']
df['amount_to_avg_spend_ratio'] = df['transaction_amount'] / df['user_avg_category_spend']

print("\n📊 Sample Features fed to the AI:")
print(df[['amount_to_income_ratio', 'amount_to_avg_spend_ratio']].head())

# ==========================================
# 3. TRAIN THE ISOLATION FOREST
# ==========================================
# We only feed the engineered ratios to the model!
X = df[['amount_to_income_ratio', 'amount_to_avg_spend_ratio']]

print("\n🧠 Training Isolation Forest...")
# contamination=0.02 tells the model we expect roughly 2% of the data to be outliers
model = IsolationForest(n_estimators=100, contamination=0.02, random_state=42)
model.fit(X)

# ==========================================
# 4. SAVE THE MODEL
# ==========================================
joblib.dump(model, 'anomaly_detector.pkl')
print("\n💾 Model saved successfully as 'anomaly_detector.pkl'!")

🚨 Starting Model 3 Training: Unsupervised Anomaly Detector (Isolation Forest)

📊 Sample Features fed to the AI:
   amount_to_income_ratio  amount_to_avg_spend_ratio
0                0.019389                   0.931729
1                0.039747                   0.816880
2                0.032260                   0.998518
3                0.024316                   1.017785
4                0.028813                   0.867376

🧠 Training Isolation Forest...

💾 Model saved successfully as 'anomaly_detector.pkl'!
